In [1]:
import pandas as pd

df = pd.read_csv("pha_cleaned.csv")
df.head()

,neo,pha,H,epoch,epoch_mjd,epoch_cal,e,a,i,om,...,class_ATE,class_CEN,class_HYA,class_IEO,class_IMB,class_MBA,class_MCA,class_OMB,class_TJN,class_TNO
0,0,0,13.679,2459000.5,59000.0,20200531.0,0.076009,2.769165,10.594067,80.305531,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,0,0,13.679,2459000.5,59000.0,20200531.0,0.229972,2.773841,24.770747,173.024741,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,0,0,13.679,2459000.5,59000.0,20200531.0,0.256936,2.668285,12.991043,169.851482,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,0,0,13.679,2459000.5,59000.0,20200531.0,0.088721,2.361418,7.141771,103.810804,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,0,0,13.679,2459000.5,59000.0,20200531.0,0.190913,2.574037,5.367427,141.571026,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [2]:
df['pha'].value_counts()

pha
0    930275
1      2066
Name: count, dtype: int64

### Handle class imbalance

In [3]:
from sklearn.model_selection import train_test_split

#### Separate features and target

In [4]:
X = df.drop(columns=['pha'])
y = df['pha']

### Split before applying SMOTE

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [6]:
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(
    sampling_strategy=1.0,
    random_state=42,
    k_neighbors=5
)

X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train_scaled, y_train
)

In [7]:
# Convert X_train_resampled into a DataFrame
X_train_resampled_df = pd.DataFrame(
    X_train_resampled,
    columns=X_train.columns
)

# Convert y_train_resampled into a Series
y_train_resampled_df = pd.Series(
    y_train_resampled,
    name='pha'
)

In [8]:
print(X_train_resampled_df.shape)
print(y_train_resampled_df.shape)


print(y_train_resampled_df.value_counts())

(1488438, 32)
(1488438,)
pha
0    744219
1    744219
Name: count, dtype: int64


In [9]:
# df.to_csv("classification.csv",index=False)

# Model Building

In [10]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

#### Initializae models

In [11]:
models = {
    'Decision Tree': DecisionTreeClassifier(
        max_depth=10,
        random_state=42,
        class_weight='balanced'
    ),

    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced'
    ),

    'XGBoost': XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    )
}

### Train and evaluate the models

In [12]:
results = []
trained_models = {}

for name, model in models.items():

    print(f"\nTraining {name}...")

    model.fit(
        X_train_resampled_df,
        y_train_resampled_df
    )

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1 Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob),
        'PR-AUC': average_precision_score(y_test, y_prob)
    })

    trained_models[name] = model

    print(classification_report(
        y_test, y_pred, zero_division=0
    ))


Training Decision Tree...
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    186056
           1       0.00      0.00      0.00       413

    accuracy                           1.00    186469
   macro avg       0.50      0.50      0.50    186469
weighted avg       1.00      1.00      1.00    186469


Training Random Forest...
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    186056
           1       0.00      0.00      0.00       413

    accuracy                           1.00    186469
   macro avg       0.50      0.50      0.50    186469
weighted avg       1.00      1.00      1.00    186469


Training XGBoost...
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    186056
           1       0.00      0.00      0.00       413

    accuracy                           1.00    186469
   macro avg       0.50      0.50      0.50    186469

### Compare all three models

In [13]:
results_df = pd.DataFrame(results)

results_df.sort_values(
    by='PR-AUC',
    ascending=False
).round(4)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC,PR-AUC
1,Random Forest,0.9978,0.0,0.0,0.0,0.5011,0.0022
0,Decision Tree,0.9978,0.0,0.0,0.0,0.5000,0.0022
2,XGBoost,0.9978,0.0,0.0,0.0,0.3342,0.0017
